#### 라이브러리 설치
필요한 라이브러리를 설치합니다.

In [ ]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets accelerate
!pip install trl peft bitsandbytes

#### 모델 로드 및 설정
1. **모델 로드**: Hugging Face의 `transformers` 라이브러리를 사용하여 사전 학습된 모델을 로드합니다.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "EleutherAI/gpt-neo-2.7B"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\28006030\.conda\envs\finetune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. **PEFT 설정**: PEFT를 사용하여 모델의 일부만 파인튜닝하도록 설정합니다.

In [2]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

model.add_adapter(peft_config)

#### 데이터 준비
파인튜닝에 사용할 데이터를 준비합니다. 예를 들어, 긍정적인 영화 리뷰 데이터를 사용할 수 있습니다.

In [3]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 442418.46 examples/s]


In [ ]:
from trl import PPOTrainer

trainer = PPOTrainer(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    reward_function=None, # reward_model 인자 제거
    ppo_config={"batch_size": 16, "epochs": 3}
)

In [8]:
from transformers import AutoTokenizer, GPTNeoForCausalLM
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler
from datasets import load_dataset
from torch.utils.data import Dataset

# 모델과 토크나이저 로드
model_name = "EleutherAI/gpt-neo-2.7B"  # 또는 사용하려는 모델 이름
model = GPTNeoForCausalLM.from_pretrained(model_name, load_in_8bit=True, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 모델을 PPOTrainer가 기대하는 형식으로 래핑
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(model)

def reward_fn(samples, **kwargs):
    rewards = []
    for sample in samples:
        reward = 1.0 if "좋아요" in sample else 0.0 
        rewards.append(reward)
    return rewards

# 데이터셋 로드 및 전처리
class IMDBDataset(Dataset):
    def __init__(self, split="train"):
        self.dataset = load_dataset("imdb", split=split)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        encoded = self.tokenizer.encode_plus(
            item['text'],
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'labels': item['label']
        }

# PPO 설정 수정
ppo_config = PPOConfig(
    batch_size=4,
    learning_rate=1e-5,
    optimize_cuda_cache=True,
    gradient_accumulation_steps=1,  # 기본값 128에서 1로 변경
    mini_batch_size=4  # batch_size와 동일하게 설정
)

# 데이터셋 인스턴스 생성
dataset = IMDBDataset("train")

# PPOTrainer 초기화
trainer = PPOTrainer(
    model=ppo_model,
    config=ppo_config,
    dataset=dataset,
    tokenizer=tokenizer
)

# 보상 함수 설정
trainer.reward_function = reward_fn

# 생성 설정
gen_kwargs = {
    "max_new_tokens": 50,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True
}

# 학습 루프
num_epochs = 1  # 테스트를 위해 에폭 수를 줄임
for epoch in range(num_epochs):
    for batch in trainer.dataloader:
        question_tensors = batch["input_ids"]
        
        # question_tensors를 리스트로 감싸서 전달
        response_tensors = trainer.generate(
            [question_tensors], 
            return_prompt=False,
            length_sampler=LengthSampler(10, 50),
            **gen_kwargs
        )
        
        rewards = trainer.reward_function(trainer.tokenizer.batch_decode(response_tensors))
        
        # question_tensors를 리스트로 감싸서 전달
        stats = trainer.step([question_tensors], response_tensors, rewards) 
        
        trainer.log_stats(stats, batch, rewards)
print("Training completed!")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


UnboundLocalError: local variable 'rows' referenced before assignment

2. **훈련 실행**: 모델을 파인튜닝합니다.

In [ ]:
trainer.train()

#### 모델 저장 및 배포
훈련이 완료된 모델을 저장하고 배포합니다.

In [ ]:
model.save_pretrained("D:\\Programming\\llama.cpp\\models")
tokenizer.save_pretrained("D:\\Programming\\llama.cpp\\models")